# TF-IDF & Logistic Regression: Abstract Domain Classification

**Goal:** Classify arXiv abstracts into their academic domain (CS, Mathematics, Physics, Statistics) using TF-IDF features and Logistic Regression, trained on the full dataset.

Pipeline: Raw abstract --> Clean text --> Tokenize --> TF-IDF vectorizer --> Feature vector --> Logistic Regression

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SEED = 67

In [37]:
from datasets import load_dataset

dataset = load_dataset("librarian-bots/arxiv-metadata-snapshot")
df = dataset["train"].select_columns(["id", "title", "abstract", "categories"]).to_pandas()
df["title"] = df["title"].str.strip()
df["abstract"] = df["abstract"].str.strip()
df["categories"] = df["categories"].str.strip()

elapsed = time.time() - start
print(f"{len(df):,} papers")

3,043,230 papers


## Domain Mapping

Each arXiv paper can have multiple categories (cs.LG stat.ML math.OC, etc). We'll use the first listed category as the primary one since this is the category the author chose as the main classification.

Rather than hardcoding every possible subcategory, we just extract the prefix from each primary category (the part before the dot) and use it to assign domains automatically. For example:
- cs.LG --> prefix cs --> domain "CS"
- math.CO --> prefix math --> domain "Math"
- stat.ML --> prefix stat --> domain "Statistics"

Physics is the exception for this since it uses many different prefixes (hep-th, cond-mat, astro-ph, etc.) that don't follow the "domain.subtopic" pattern, so for that we'll just maintain a set of known physics prefixes.

In [42]:
#primary category
df["primary_category"] = df["categories"].str.split().str[0]

#prefix extraction (prefix.subcategory)
df["prefix"] = df["primary_category"].apply(lambda x: x.split(".")[0] if "." in x else x)

print(f"Total unique prefixes: {df['prefix'].nunique()}")

#prefix distribution
prefix_counts = df["prefix"].value_counts()
for prefix, count in prefix_counts.items():
    print(f"{prefix:10s} {count:>10,}")

Total unique prefixes: 38
cs            763,466
math          578,989
cond-mat      348,201
astro-ph      335,204
physics       210,414
hep-ph        142,592
quant-ph      130,912
hep-th        112,879
eess           72,739
gr-qc          71,477
stat           62,110
nucl-th        35,641
math-ph        34,147
q-bio          34,125
hep-ex         25,051
nlin           20,759
hep-lat        19,101
q-fin          13,483
nucl-ex        12,410
econ           11,327
chao-dyn        1,770
alg-geom        1,209
q-alg           1,177
cmp-lg            894
solv-int          844
dg-ga             562
patt-sol          452
funct-an          320
adap-org          306
mtrl-th           165
comp-gas          140
chem-ph           129
supr-con           69
atom-ph            68
acc-phys           46
plasm-ph           28
ao-sci             13
bayes-an           11


CS, Math and Stats all follow the prefix.subtopic pattern, so we can catch them automatically by checking just the prefix. Whereas for physics, there is a bunch of standalone prefixes for it, so these are maintained in an explicit set.

In [45]:
#domain mapping rules for physics

PHYSICS_PREFIXES = {
    "hep-th",    #High Energy Physics - Theory
    "hep-ph",    #High Energy Physics - Phenomenology
    "hep-lat",   #High Energy Physics - Lattice
    "hep-ex",    #High Energy Physics - Experiment
    "cond-mat",  #Condensed Matter
    "astro-ph",  #Astrophysics
    "gr-qc",     #General Relativity and Quantum Cosmology
    "quant-ph",  #Quantum Physics
    "nucl-th",   #Nuclear Theory
    "nucl-ex",   #Nuclear Experiment
    "physics",   #General Physics (physics.optics, physics.flu-dyn, etc.)
}


#map function to map prefixes to domains
def map_prefix_to_domain(prefix):
    if prefix == "cs":                  return "CS"
    elif prefix in {"math", "math-ph"}: return "Math"
    elif prefix == "stat":              return "Statistics"
    elif prefix in PHYSICS_PREFIXES:    return "Physics"
    else:                               return None  # drop eess, econ, q-fin, nlin, q-bio, etc.

df["domain"] = df["prefix"].apply(map_prefix_to_domain)

#mapped and dropped papers
mapped = df[df["domain"].notna()]
dropped = df[df["domain"].isna()]

print(f"# of Papers mapped to a domain: {len(mapped):,}")
print(f"# of Papers dropped           : {len(dropped):,}")
print(f"\nDropped prefixes:")
print(dropped["prefix"].value_counts().to_string())

# of Papers mapped to a domain: 2,882,594
# of Papers dropped           : 160,636

Dropped prefixes:
prefix
eess        72739
q-bio       34125
nlin        20759
q-fin       13483
econ        11327
chao-dyn     1770
alg-geom     1209
q-alg        1177
cmp-lg        894
solv-int      844
dg-ga         562
patt-sol      452
funct-an      320
adap-org      306
mtrl-th       165
comp-gas      140
chem-ph       129
supr-con       69
atom-ph        68
acc-phys       46
plasm-ph       28
ao-sci         13
bayes-an       11


sanity check, verify mapping

In [85]:
#verify the mapping: show which primary categories ended up in each domain
for domain in sorted(mapped["domain"].unique()):
    domain_df = mapped[mapped["domain"] == domain]
    prefixes_in_domain = sorted(domain_df["prefix"].unique())
    subcategories = sorted(domain_df["primary_category"].unique())
    print(f"{domain}: {len(domain_df):,} papers")
    print(f"Prefixes: {prefixes_in_domain}")
    print(f"Subcategories ({len(subcategories)}): {subcategories[:10]}{' ' if len(subcategories) > 10 else ''}")
    print()


CS: 763,466 papers
Prefixes: ['cs']
Subcategories (40): ['cs.AI', 'cs.AR', 'cs.CC', 'cs.CE', 'cs.CG', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DB'] 

Math: 613,136 papers
Prefixes: ['math', 'math-ph']
Subcategories (31): ['math-ph', 'math.AC', 'math.AG', 'math.AP', 'math.AT', 'math.CA', 'math.CO', 'math.CT', 'math.CV', 'math.DG'] 

Physics: 1,443,882 papers
Prefixes: ['astro-ph', 'cond-mat', 'gr-qc', 'hep-ex', 'hep-lat', 'hep-ph', 'hep-th', 'nucl-ex', 'nucl-th', 'physics', 'quant-ph']
Subcategories (47): ['astro-ph', 'astro-ph.CO', 'astro-ph.EP', 'astro-ph.GA', 'astro-ph.HE', 'astro-ph.IM', 'astro-ph.SR', 'cond-mat', 'cond-mat.dis-nn', 'cond-mat.mes-hall'] 

Statistics: 62,110 papers
Prefixes: ['stat']
Subcategories (5): ['stat.AP', 'stat.CO', 'stat.ME', 'stat.ML', 'stat.OT']



drop any papers w/out a domain, keeping the big 4 (CS, Math, Physics, Stats)

In [47]:
df = df[df["domain"].notna()].reset_index(drop=True)

print(f"Final dataset: {len(df):,} papers")
print(f"\nDomain distribution:")
domain_counts = df["domain"].value_counts()
for domain, count in domain_counts.items():
    pct = count / len(df) * 100
    print(f"  {domain:12s} {count:>10,}  ({pct:.1f}%)")

Final dataset: 2,882,594 papers

Domain distribution:
  Physics       1,443,882  (50.1%)
  CS              763,466  (26.5%)
  Math            613,136  (21.3%)
  Statistics       62,110  (2.2%)


Cleaning

In [48]:
LATEX_PATTERNS = [
    (r"\$[^$]*\$", ""),
    (r"\\[a-zA-Z]+\{[^}]*\}", ""),
    (r"\\[a-zA-Z]+", ""),
    (r"\{[^}]*\}", ""),
    (r"\s+", " "),
]

def clean_text(text):
    if not isinstance(text, str):
        return ""
    for pattern, replacement in LATEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    text = text.encode("ascii", errors="ignore").decode()
    return text.strip().lower()


df["abstract_clean"] = df["abstract"].apply(clean_text)

## Train/Test Split

80/10/10 split w/ stratification to preserve class proportions

In [49]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["domain"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["domain"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

CLASSES = sorted(df["domain"].unique().tolist())

print(f"Train: {len(train_df):,}")
print(f"Val: {len(val_df):,}")
print(f"Test: {len(test_df):,}")
print(f"Classes: {CLASSES}")

Train: 2,306,075
Val: 288,259
Test: 288,260
Classes: ['CS', 'Math', 'Physics', 'Statistics']


Sampling

In [17]:
for domain in CLASSES:
    sample = train_df[train_df["domain"] == domain].iloc[0]
    print(f"{domain}:")
    print(f"Category: {sample['primary_category']}")
    print(f"Title   : {sample['title']}")
    print(f"Abstract: {sample['abstract_clean'][:200]}...")
    print()

CS:
Category: cs.CV
Title   : Simple is not Easy: A Simple Strong Baseline for TextVQA and TextCaps
Abstract: texts appearing in daily scenes that can be recognized by ocr (optical character recognition) tools contain significant information, such as street name, product brand and prices. two tasks -- text-ba...

Math:
Category: math.AG
Title   : On fundamental groups related to the Hirzebruch surface F_1
Abstract: given a projective surface and a generic projection to the plane, the braid monodromy factorization (and thus, the braid monodromy type) of the complement of its branch curve is one of the most import...

Physics:
Category: astro-ph
Title   : Maximum Likelihood Analysis of Clusters of Ultra-High Energy Cosmic Rays
Abstract: we present a numerical code designed to conduct a likelihood analysis for clusters of nucleons above 10**19 ev originating from discrete astrophysical sources such as powerful radio galaxies, gamma-ra...

Statistics:
Category: stat.ME
Title   : A flexible

## TF-IDF Vectorization

In [51]:
vectorizer = TfidfVectorizer(
    max_features=20_000,     #vocabulary size
    ngram_range=(1, 2),      #unigrams & bigrams
    sublinear_tf=True,       #1 + log(TF) to mitigate large TF dominance
    min_df=5,                #ignore words in fewer than 5 docs (stricter for large dataset)
    stop_words="english"     #remove common English words
)

#our unigram will take care of single words like algorithm,theorem, quantum, regression, etc.
#while our bigram can capture phrases like neural network, machine learning,, random variables, black hole, etc.

#fitting TF-IDF on training data
X_train = vectorizer.fit_transform(train_df["abstract_clean"])
X_val = vectorizer.transform(val_df["abstract_clean"])
X_test = vectorizer.transform(test_df["abstract_clean"])

y_train = train_df["domain"].values
y_val = val_df["domain"].values
y_test = test_df["domain"].values


In [52]:
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,} features")
print(f"Feature matrix shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_val  : {X_val.shape}")
print(f"X_test : {X_test.shape}")
print(f"\nSparsity: {X_train.nnz:,} non-zero entries out of {X_train.shape[0] * X_train.shape[1]:,} total ({X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.2f}% non-zero)")

Vocabulary size: 20,000 features
Feature matrix shapes:
X_train: (2306075, 20000)
X_val  : (288259, 20000)
X_test : (288260, 20000)

Sparsity: 162,033,325 non-zero entries out of 46,121,500,000 total (0.35% non-zero)


How a single abstract looks like after transformation (the actual feature vector the classifier will see)

In [53]:
feature_names = vectorizer.get_feature_names_out()

for domain in CLASSES:
    domain_index = train_df[train_df["domain"] == domain].index[0]
    tfidf_vector = X_train[domain_index]
    scores = tfidf_vector.toarray()[0]
    top_10_indices = scores.argsort()[-10:][::-1]

    print(f"{domain}: ")
    print(f"Abstract: {train_df.iloc[domain_index]['abstract_clean'][:100]}...")
    print(f"Non-zero features: {tfidf_vector.nnz} out of {len(feature_names):,}")
    for rank, idx in enumerate(top_10_indices, 1):
        print(f"    {rank:2d}. '{feature_names[idx]}' = {scores[idx]:.4f}")
    print()

CS: 
Abstract: texts appearing in daily scenes that can be recognized by ocr (optical character recognition) tools ...
Non-zero features: 120 out of 20,000
     1. 'ocr' = 0.2392
     2. 'text based' = 0.2180
     3. 'image captioning' = 0.1909
     4. 'based image' = 0.1759
     5. 'captioning' = 0.1753
     6. 'text' = 0.1682
     7. 'sota' = 0.1530
     8. 'modality' = 0.1455
     9. 'encoding' = 0.1318
    10. 'popular' = 0.1174

Math: 
Abstract: given a projective surface and a generic projection to the plane, the braid monodromy factorization ...
Non-zero features: 32 out of 20,000
     1. 'braid' = 0.3241
     2. 'monodromy' = 0.3174
     3. 'solvable' = 0.2671
     4. 'complement' = 0.2580
     5. 'factorization' = 0.2561
     6. 'branch' = 0.2452
     7. 'curve' = 0.2154
     8. 'f_' = 0.2087
     9. 'topological invariants' = 0.2055
    10. 'fundamental group' = 0.1880

Physics: 
Abstract: we present a numerical code designed to conduct a likelihood analysis for clusters of n

IDF values:

In [60]:
idf_scores = vectorizer.idf_
word_idf_pairs = list(zip(feature_names, idf_scores))
word_idf_sorted = sorted(word_idf_pairs, key=lambda x: x[1])

print("Lowest IDF:") #appear everywhere
for word, idf in word_idf_sorted[:10]:
    print(f"  '{word}' IDF = {idf:.4f}")

print()

print(f"Highest IDF:") #appear in few documents
for word, idf in word_idf_sorted[-10:]:
    print(f"  '{word}' IDF = {idf:.4f}")

Lowest IDF:
  'results' IDF = 2.3002
  'model' IDF = 2.3163
  'using' IDF = 2.3834
  'based' IDF = 2.4389
  'paper' IDF = 2.5064
  'data' IDF = 2.6050
  'study' IDF = 2.6250
  'time' IDF = 2.7238
  'models' IDF = 2.7584
  'present' IDF = 2.7941

Highest IDF:
  'cir' IDF = 9.5374
  'metaverse' IDF = 9.5463
  'lbgs' IDF = 9.5530
  'msps' IDF = 9.5575
  'bcgs' IDF = 9.5780
  'etgs' IDF = 9.5873
  'garment' IDF = 9.6061
  'mbl' IDF = 9.6349
  'tta' IDF = 9.6546
  'smgs' IDF = 9.8251


In [65]:
# IDF for domain-specific words
domain_words = {
    "Physics":     ["hamiltonian", "lattice", "spin", "gauge", "quark", "boson"],
    "CS":          ["algorithm", "complexity", "optimization", "dataset", "neural network"],
    "Math":        ["theorem", "conjecture", "manifold", "proof", "algebraic"],
    "Statistics":  ["estimator", "regression", "bayesian", "variance", "posterior"],
}

print("IDF scores for domain-specific words:\n")
for domain, words in domain_words.items():
    print(f"{domain}:")
    for word in words:
        if word in vectorizer.vocabulary_:
            idx = vectorizer.vocabulary_[word]
            print(f"    {word:20s} IDF = {idf_scores[idx]:.4f}")
        else:
            print(f"    {word:20s} (not in vocabulary)")
    print()

IDF scores for domain-specific words:

Physics:
    hamiltonian          IDF = 5.0090
    lattice              IDF = 4.3753
    spin                 IDF = 3.9641
    gauge                IDF = 4.6851
    quark                IDF = 5.0350
    boson                IDF = 5.6076

CS:
    algorithm            IDF = 3.6881
    complexity           IDF = 4.4659
    optimization         IDF = 4.2634
    dataset              IDF = 4.1918
    neural network       IDF = 4.9508

Math:
    theorem              IDF = 4.5032
    conjecture           IDF = 5.0143
    manifold             IDF = 5.1657
    proof                IDF = 4.6342
    algebraic            IDF = 5.0965

Statistics:
    estimator            IDF = 5.9299
    regression           IDF = 5.4044
    bayesian             IDF = 5.3591
    variance             IDF = 5.6193
    posterior            IDF = 6.3282



## Logistic Regression

Logistic Regression learns a weight for every feature for every class.

The model takes the dot product of the document's TF-IDF vector with each class's weight vector. 

If the "Physics" weights assign high positive values to "Hamiltonian" and "spin," and the document has high TF-IDF scores for those words, then the dot product is large => high probability of Physics.

And training minimizes cross-entropy loss via gradient descent (solver = "saga")

In [66]:
classifier = LogisticRegression(
    C=1.0,                      #regularization strength
    max_iter=1000,              #maximum optimization iterations
    multi_class="multinomial",  #softmax for multi-class
    solver="saga",              #stochastic gradient descent variant
    random_state=SEED,
    n_jobs=-1
)

#training our logistic regression classifier on the full training dataset:
classifier.fit(X_train, y_train)

train_accuracy = classifier.score(X_train, y_train)
val_accuracy   = classifier.score(X_val, y_val)
test_accuracy  = classifier.score(X_test, y_test)

#accuracy results
print(f"Train     : {train_accuracy:.4f}")
print(f"Validation: {val_accuracy:.4f}")
print(f"Test      : {test_accuracy:.4f}")

c:\Users\Widi\anaconda3\envs\nlp\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Train     : 0.9506
Validation: 0.9470
Test      : 0.9469


Learned Weights: 

The weight vector w_c tells us which features push toward class c. High positive weights imply that the feature strongly signals that class

In [86]:
print(f"Weight matrix shape: {classifier.coef_.shape}")
print(f"({len(CLASSES)} classes x {classifier.coef_.shape[1]:,} features)")
print(f"Bias vector: {dict(zip(classifier.classes_, classifier.intercept_.round(4)))}")

Weight matrix shape: (4, 20000)
(4 classes x 20,000 features)
Bias vector: {'CS': np.float64(-0.0835), 'Math': np.float64(1.5506), 'Physics': np.float64(0.7578), 'Statistics': np.float64(-2.2249)}


In [87]:
#top 15 most predictive features per class
for class_index, class_name in enumerate(classifier.classes_):
    weights = classifier.coef_[class_index]
    top_indices = weights.argsort()[-15:][::-1]

    print(f"{class_name}:")
    pairs = [f"'{feature_names[i]}' ({weights[i]:+.3f})" for i in top_indices]
    print(f"{', '.join(pairs[:8])}")
    print(f"{', '.join(pairs[8:])}")
    print()

CS:
'robot' (+6.784), 'robots' (+5.435), 'software engineering' (+5.372), 'automata' (+5.170), 'robotic' (+5.029), 'experimental results' (+4.669), 'ai' (+4.627), 'learning' (+4.464)
'codes' (+4.406), 'mimo' (+4.298), 'neural' (+4.227), 'extensive experiments' (+4.226), 'stoc' (+4.225), 'semantics' (+4.199), 'wireless' (+4.187)

Math:
'numerical experiments' (+6.205), 'math' (+5.535), 'mathematics' (+4.348), 'mixed integer' (+4.301), 'computational experiments' (+4.199), 'prove' (+4.155), 'let' (+4.012), 'weak solutions' (+3.722)
'posedness' (+3.572), 'asymptotics' (+3.378), 'mathematical' (+3.341), 'simulation study' (+3.319), 'power flow' (+3.219), 'numerical tests' (+3.190), 'steady states' (+3.176)

Physics:
'quantum' (+13.895), 'cosmological' (+8.331), 'astronomical' (+8.236), 'qcd' (+7.289), 'physics' (+7.276), 'galaxy' (+7.185), 'astrophysical' (+6.738), 'astronomy' (+6.735)
'qubit' (+6.703), 'telescope' (+6.351), 'stellar' (+6.327), 'gravitational' (+6.098), 'galaxies' (+5.986)

### Walkthrough

Tracing one abstract through the full pipeline: raw text --> TF-IDF --> dot product --> softmax --> prediction.

In [ ]:
sample_row = test_df.iloc[0]
sample_text = sample_row["abstract_clean"]
sample_label = sample_row["domain"]

#sample print
print(f"Abstract: {sample_text[:200]}...")
print(f"True label: {sample_label}")
print()

#step 1: TF-IDF
sample_tfidf = vectorizer.transform([sample_text])
sample_dense = sample_tfidf.toarray()[0]

print(f"Vector: ({len(sample_dense):,},) — one score per vocabulary word")
print(f"Non-zero features: {np.count_nonzero(sample_dense)}")
top_5 = sample_dense.argsort()[-5:][::-1]
print(f"Top 5: {[(feature_names[i], round(sample_dense[i], 4)) for i in top_5]}")
print()

#step 2: Dot product w * x + b
raw_scores = classifier.decision_function(sample_tfidf)[0]
for name, score in zip(classifier.classes_, raw_scores):
    print(f"  {name:12s}: {score:+.4f}")
print()

#step 3: Softmax
probs = classifier.predict_proba(sample_tfidf)[0]
for name, prob in zip(classifier.classes_, probs):
    print(f"  {name:12s}: {prob:.4f}")
print()

#step 4: Predict
predicted = classifier.classes_[probs.argmax()]
print(f"Prediction: {predicted} (confidence: {probs.max():.4f})")
print(f"True: {sample_label} | {'Correct' if predicted == sample_label else 'Wrong'}")

Abstract: in this paper we assume quantum dots can be assimilated to fermi hubbard sites when the coulomb interaction between electrons is higher compared to their tunneling. the study of pairwise entanglement ...
True label: Physics

Vector: (20,000,) — one score per vocabulary word
Non-zero features: 62
Top 5: [('coulomb interaction', np.float64(0.2987)), ('coulomb', np.float64(0.2331)), ('quantum dots', np.float64(0.2258)), ('dots', np.float64(0.2205)), ('entanglement', np.float64(0.2157))]

  CS          : -3.0427
  Math        : -1.1579
  Physics     : +8.4997
  Statistics  : -4.2992

  CS          : 0.0000
  Math        : 0.0001
  Physics     : 0.9999
  Statistics  : 0.0000

Prediction: Physics (confidence: 0.9999)
True: Physics | Correct



## Evalulation
Classification Report

In [79]:
y_predicted = classifier.predict(X_test)

print(classification_report(y_test, y_predicted, target_names=CLASSES, digits=4))

              precision    recall  f1-score   support

          CS     0.9322    0.9450    0.9386     76347
        Math     0.9191    0.9238    0.9215     61313
     Physics     0.9735    0.9729    0.9732    144389
  Statistics     0.7443    0.5906    0.6586      6211

    accuracy                         0.9469    288260
   macro avg     0.8923    0.8581    0.8730    288260
weighted avg     0.9461    0.9469    0.9463    288260



## Sample Predictions

In [83]:
for i in range(12):
    row = test_df.iloc[i]
    sample_tfidf = vectorizer.transform([row["abstract_clean"]])
    predicted  = classifier.predict(sample_tfidf)[0]
    confidence = classifier.predict_proba(sample_tfidf)[0].max()

    match = "correct" if predicted == row["domain"] else "wrong"
    print(f"[{i+1}] {match}")
    print(f"  Category : {row['primary_category']}")
    print(f"  Abstract : {row['abstract'][:120]}...")
    print(f"  True     : {row['domain']}")
    print(f"  Predicted: {predicted} (confidence: {confidence:.4f})")
    print()

[1] correct
  Category : quant-ph
  Abstract : In this paper we assume quantum dots can be assimilated to Fermi Hubbard
sites when the Coulomb interaction between elec...
  True     : Physics
  Predicted: Physics (confidence: 0.9999)

[2] correct
  Category : cs.CR
  Abstract : Improved interoperability between public and private organizations is of key
significance to make digital government new...
  True     : CS
  Predicted: CS (confidence: 0.9901)

[3] correct
  Category : cs.CL
  Abstract : This paper presents the development and validation of an eye-tracking dataset designed to investigate how second-languag...
  True     : CS
  Predicted: CS (confidence: 0.9978)

[4] correct
  Category : cond-mat.mes-hall
  Abstract : Topology is familiar mostly from mathematics, but also natural sciences have
found its concepts useful. Those concepts h...
  True     : Physics
  Predicted: Physics (confidence: 0.9829)

[5] correct
  Category : cond-mat.mes-hall
  Abstract : We study the magnetoe

In [90]:
#misclassified examples
wrong_mask = y_predicted != y_test
wrong_indices = np.where(wrong_mask)[0]

#total misclassified
print(f"{len(wrong_indices):,} out of {len(y_test):,} ({len(wrong_indices)/len(y_test)*100:.1f}%)\n")

#which pairs get confused most
error_pairs = Counter()
for i in wrong_indices:
    error_pairs[(y_test[i], y_predicted[i])] += 1

#the most common confusions:
for (true, pred), count in error_pairs.most_common(10):
    print(f"  {true:12s} --> predicted as {pred:12s}  ({count:,} times)")

15,316 out of 288,260 (5.3%)

  CS           --> predicted as Math          (2,246 times)
  Math         --> predicted as Physics       (2,223 times)
  Physics      --> predicted as Math          (2,131 times)
  Math         --> predicted as CS            (1,908 times)
  Statistics   --> predicted as CS            (1,698 times)
  Physics      --> predicted as CS            (1,644 times)
  CS           --> predicted as Physics       (1,361 times)
  Statistics   --> predicted as Math          (610 times)
  CS           --> predicted as Statistics    (590 times)
  Math         --> predicted as Statistics    (538 times)


In [35]:
#misclassified examples with probability distributions
print("\nSample misclassified papers:\n")
for i in wrong_indices[:5]:
    row = test_df.iloc[i]
    sample_tfidf = vectorizer.transform([row["abstract_clean"]])
    probs = classifier.predict_proba(sample_tfidf)[0]
    predicted = classifier.classes_[probs.argmax()]

    print(f"  Category : {row['primary_category']}")
    print(f"  Abstract : {row['abstract'][:150]}...")
    print(f"  True     : {row['domain']}")
    print(f"  Predicted: {predicted}")
    print(f"  Probs    : {dict(zip(classifier.classes_, probs.round(4)))}")
    print()


Sample misclassified papers:

  Category : gr-qc
  Abstract : It is shown that an $(n+1)$-dimensional asymptotically anti-de Sitter
solution of the Einstein-vacuum equations is locally isometric to pure anti-de
S...
  True     : Physics
  Predicted: Math
  Probs    : {'CS': np.float64(0.0005), 'Math': np.float64(0.9268), 'Physics': np.float64(0.0727), 'Statistics': np.float64(0.0001)}

  Category : stat.ML
  Abstract : Top-performing deep architectures are trained on massive amounts of labeled
data. In the absence of labeled data for a certain task, domain adaptation...
  True     : Statistics
  Predicted: CS
  Probs    : {'CS': np.float64(0.9498), 'Math': np.float64(0.0005), 'Physics': np.float64(0.0133), 'Statistics': np.float64(0.0364)}

  Category : cs.IT
  Abstract : We present new efficient recursive decoders for the Barnes-Wall lattices
based on their squaring construction. The analysis of the new decoders reveal...
  True     : CS
  Predicted: Physics
  Probs    : {'CS': np.fl

These confusions reflect genuine domain overlap rather then model failure. If we look at the original dataset, most papers fall under multiple categories and will most likely share similar language.